In [2]:
# -*- coding: utf-8 -*-
"""Contas_final.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Ac0I2UOtV5VseYErtHwCS2Qpjl9DG745
"""

# ============================================================
# CONFIGURAÇÃO DE FUSO HORÁRIO - BRASÍLIA
# ============================================================
import pytz
from datetime import datetime

fuso_br = pytz.timezone('America/Sao_Paulo')

def agora_brasilia():
    return datetime.now(pytz.timezone('America/Sao_Paulo'))

# ============================================================
# IMPORTAÇÕES
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import re
import numpy as np
import json as _json
import openpyxl

!pip install --quiet --upgrade gspread
!pip install pytz

# ============================================================
# MONTAR DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def formatar_valor_br(valor):
    if pd.isna(valor) or valor == 0:
        return "R$ 0,00"

    valor = float(valor)
    inteiro = int(abs(valor))
    decimal = int(round((abs(valor) - inteiro) * 100))
    inteiro_str = f"{inteiro:,}".replace(',', '.')
    sinal = "-" if valor < 0 else ""

    return f"{sinal}R$ {inteiro_str},{decimal:02d}"


def extrair_parcela(parcela_str):
    if pd.isna(parcela_str) or str(parcela_str).strip() in ('', '-'):
        return None, None

    s = str(parcela_str).strip()

    m = re.search(r'(\d+)\s*de\s*(\d+)', s)
    if m:
        return int(m.group(1)), int(m.group(2))

    m = re.search(r'(\d+)/(\d+)', s)
    if m:
        return int(m.group(1)), int(m.group(2))

    return None, None


def limpar_valor(serie):
    s = serie.astype(str)
    s = s.str.replace('R$', '', regex=False)
    s = s.str.replace(' ', '', regex=False)
    s = s.str.replace('.', '', regex=False)
    s = s.str.replace(',', '.', regex=False)

    return pd.to_numeric(s, errors='coerce')


def achar_coluna(df, opcoes):
    for col in df.columns:
        cl = str(col).lower()

        if any(op.lower() in cl for op in opcoes):
            return col

    return None


def nome_mes_pt(numero):
    meses = {
        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    }

    return meses.get(numero, "")


pasta = "/content/drive/MyDrive/Contas/Faturas"
pasta_casa = "/content/drive/MyDrive/Contas/Casa"
pasta_relatorios = "/content/drive/MyDrive/Contas/Relatorios"
os.makedirs(pasta_relatorios, exist_ok=True)
arquivos = sorted(os.listdir(pasta))
print(f"📁 Encontrados {len(arquivos)} arquivos na pasta de Faturas")

arquivos_pasta_casa = sorted(os.listdir(pasta_casa)) if os.path.exists(pasta_casa) else []
print(f"📁 Encontrados {len(arquivos_pasta_casa)} arquivos na pasta de Casa ({pasta_casa})")
for _a in arquivos_pasta_casa:
    print(f"   📄 {_a}")

# ============================================================
# ARQUIVO ÚNICO DE FATURAS
# ============================================================

arquivo_faturas = os.path.join(pasta, "Faturas.xlsx")

if not os.path.exists(arquivo_faturas):
    raise FileNotFoundError(
        f"Arquivo de faturas não encontrado:\n{arquivo_faturas}"
    )

# Carrega somente a aba FATURAS
df_faturas_origem = pd.read_excel(
    arquivo_faturas,
    sheet_name="FATURAS",
    engine="openpyxl"
)

df_faturas_origem.columns = df_faturas_origem.columns.str.strip()

# Verificar coluna obrigatória
if "Source.Name" not in df_faturas_origem.columns:
    raise ValueError(
        "A aba FATURAS precisa possuir a coluna 'Source.Name'."
    )

print("\n" + "="*60)
print("📋 ARQUIVO DE FATURAS")
print("="*60)

print(f"📄 Arquivo: {os.path.basename(arquivo_faturas)}")
print(f"📊 Registros encontrados: {len(df_faturas_origem)}")

# Lista das faturas existentes dentro do arquivo
faturas_encontradas = sorted(
    df_faturas_origem["Source.Name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

faturas_2025 = sorted(
    [f for f in faturas_encontradas if re.search(r'Fatura2025-', f)]
)

faturas_2026 = sorted(
    [f for f in faturas_encontradas if re.search(r'Fatura2026-', f)]
)

print(f"\n📁 Faturas 2025 encontradas: {len(faturas_2025)}")
for f in faturas_2025:
    print(f"   📄 {f}")

print(f"\n📁 Faturas 2026 encontradas: {len(faturas_2026)}")
for f in faturas_2026:
    print(f"   📄 {f}")

print(f"\n📊 Total de faturas: {len(faturas_encontradas)}")
print("="*60)


# ============================================================
# LEITURA DE PLANILHAS
# ============================================================
import json as _json
import openpyxl

_gspread_client = None
def _get_gspread_client():
    global _gspread_client
    if _gspread_client is None:
        from google.colab import auth as _auth
        _auth.authenticate_user()
        import gspread
        from google.auth import default as _default
        creds, _ = _default()
        _gspread_client = gspread.authorize(creds)
    return _gspread_client

def ler_planilha_google(titulo):
    gc = _get_gspread_client()
    try:
        planilha = gc.open(titulo)
    except:
        # Se não encontrar, tentar sem o "(1)"
        titulo_sem_sufixo = re.sub(r'\s*\(\d+\)\s*$', '', titulo)
        if titulo_sem_sufixo != titulo:
            planilha = gc.open(titulo_sem_sufixo)
        else:
            raise
    aba = planilha.sheet1
    valores = aba.get_all_values()
    df = pd.DataFrame(valores[1:], columns=valores[0])
    return df

def ler_fatura(pasta, nome_base_sem_ext):
    caminho_csv = os.path.join(pasta, nome_base_sem_ext + '.csv')
    caminho_xlsx = os.path.join(pasta, nome_base_sem_ext + '.xlsx')

    try:
        return ler_planilha_google(nome_base_sem_ext), 'gsheet (ao vivo)'
    except Exception as e:
        print(f"   ⚠ Não consegui abrir a Planilha Google '{nome_base_sem_ext}' ({str(e)[:50]}), tentando arquivo local...")

    try:
        df = pd.read_csv(caminho_csv, sep=';', encoding='utf-8-sig')
        if df.shape[1] == 1:
            raise ValueError("sep errado")
        return df, 'csv (estático)'
    except Exception:
        pass

    try:
        df = pd.read_excel(caminho_xlsx, engine='openpyxl')
        return df, 'xlsx (estático)'
    except Exception as e:
        print(f"   ⚠ Erro ao ler .xlsx: {str(e)[:50]}")
        pass

    try:
        df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig')
        return df, 'csv (estático)'
    except Exception as e:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para {nome_base_sem_ext} (tentado: .gsheet, .csv, .xlsx)")

def ler_despesa_casa(pasta, nome_arquivo):
    caminho = os.path.join(pasta, nome_arquivo)

    if nome_arquivo.endswith('.gsheet'):
        try:
            nome_base = os.path.splitext(nome_arquivo)[0]
            gc = _get_gspread_client()
            planilha = gc.open(nome_base)
            aba = planilha.sheet1
            valores = aba.get_all_values()
            df = pd.DataFrame(valores[1:], columns=valores[0])
            return df, 'gsheet (ao vivo)'
        except Exception as e:
            print(f"   ⚠ Erro ao abrir .gsheet: {str(e)[:50]}")
            raise

    if nome_arquivo.endswith('.xlsx'):
        # lê sempre a primeira aba, independente do nome dela —
        # algumas planilhas de casa têm a aba com nome de mês errado
        # (ex.: copiada de um mês anterior e não renomeada)
        df = pd.read_excel(caminho, sheet_name=0)
        return df, 'xlsx'

    try:
        df = pd.read_csv(caminho, sep=';', encoding='utf-8-sig')
        return df, 'csv'
    except Exception:
        try:
            df = pd.read_csv(caminho, sep=',', encoding='utf-8-sig')
            return df, 'csv'
        except Exception as e:
            raise FileNotFoundError(f"Não foi possível ler {nome_arquivo}")

# ============================================================
# 1. CARREGAR FATURAS 2026
# ============================================================
print("\n" + "="*60)
print("💳 CARREGANDO FATURAS 2026...")
print("="*60)

nomes_base_2026 = sorted(
    faturas_2026
)

lista_faturas = []
total_cancelado = 0
total_jornal = 0
total_pagamento = 0
total_negativo = 0
total_bruto_real = 0
transacoes_canceladas = []

# Dicionário para guardar o mês de cada fatura
mes_fatura_por_arquivo = {}

for nome_arquivo_fatura in nomes_base_2026:

    try:

        # --------------------------------------------------------
        # Selecionar somente os registros desta fatura
        # --------------------------------------------------------
        df = df_faturas_origem[
            df_faturas_origem["Source.Name"].astype(str).str.strip()
            == nome_arquivo_fatura
        ].copy()

        print(
            f"\n📄 Processando: "
            f"{nome_arquivo_fatura}  [Faturas.xlsx]"
        )

        if df.empty:
            print(f"  ⚠ Nenhum registro encontrado.")
            continue

        df.columns = df.columns.str.strip()

        col_estab = achar_coluna(
            df,
            ['estabelecimento', 'descrição', 'descricao']
        )

        # Preferir TIPO_AUTO; recorrer a TIPO quando necessário.
        col_tipo_original = achar_coluna(df, ['tipo', 'categoria'])
        col_tipo_auto = achar_coluna(df, ['tipo_auto'])
        if col_tipo_auto is not None or col_tipo_original is not None:
            auto = (df[col_tipo_auto].fillna('').astype(str).str.strip()
                    if col_tipo_auto is not None else pd.Series('', index=df.index))
            original = (df[col_tipo_original].fillna('').astype(str).str.strip()
                        if col_tipo_original is not None else pd.Series('', index=df.index))
            valido = ~auto.str.upper().isin(['', 'A CLASSIFICAR', 'NAN', 'NONE', 'NULL'])
            df['_tipo_relatorio'] = auto.where(valido, original).replace('', 'A CLASSIFICAR')
            col_tipo = '_tipo_relatorio'
        else:
            col_tipo = None

        col_valor = achar_coluna(
            df,
            ['valor']
        )

        col_parcela = achar_coluna(
            df,
            ['parcela']
        )

        if col_valor is None:
            print(
                f"  ⚠ Pulando '{nome_arquivo_fatura}': "
                f"não encontrei coluna de valor."
            )
            continue

        # --------------------------------------------------------
        # EXTRAIR MÊS DA FATURA DO Source.Name
        # --------------------------------------------------------
        match = re.search(
            r'Fatura(\d{4})-(\d{2})-(\d{2})',
            nome_arquivo_fatura
        )

        if match:

            ano = int(match.group(1))
            mes = int(match.group(2))
            dia = int(match.group(3))

            mes_fatura_por_arquivo[nome_arquivo_fatura] = mes

        else:

            match = re.search(
                r'Fatura(\d{4})-(\d{2})',
                nome_arquivo_fatura
            )

            if match:

                ano = int(match.group(1))
                mes = int(match.group(2))
                dia = 1

                mes_fatura_por_arquivo[nome_arquivo_fatura] = mes

            else:

                print(
                    f"  ⚠ Pulando '{nome_arquivo_fatura}': "
                    f"não foi possível extrair data."
                )
                continue

        # --------------------------------------------------------
        # DADOS DA FATURA
        # --------------------------------------------------------
        df['mes_ano'] = f"{mes:02d}/{ano}"
        df['data_fatura'] = pd.Timestamp(
            ano,
            mes,
            dia
        )
        df['ano'] = ano
        df['mes'] = mes
        df['mes_fatura'] = mes
        df['nome_fatura'] = nome_arquivo_fatura

        df['valor'] = limpar_valor(df[col_valor])
        total_bruto_real += df['valor'].sum()

        # ========================================================
        # CANCELADOS
        # ========================================================
        if col_tipo is not None:

            mask_cancelado = (
                df[col_tipo]
                .astype(str)
                .str.upper()
                .str.contains(
                    'CANCELADO',
                    na=False
                )
            )

            if mask_cancelado.any():

                valor_cancelado = (
                    df.loc[
                        mask_cancelado,
                        'valor'
                    ].sum()
                )

                total_cancelado += (
                    abs(valor_cancelado)
                    if valor_cancelado < 0
                    else valor_cancelado
                )

                transacoes_canceladas.extend(
                    df.loc[
                        mask_cancelado,
                        col_estab
                    ].tolist()
                    if col_estab
                    else ['N/A']
                )

                print(
                    f"  📌 Registrados "
                    f"{mask_cancelado.sum()} cancelados: "
                    f"{formatar_valor_br(abs(valor_cancelado))}"
                )

            # ====================================================
            # JORNAL
            # ====================================================
            mask_jornal = (
                df[col_tipo]
                .astype(str)
                .str.upper()
                .str.contains(
                    'JORNAL',
                    na=False
                )
            )

            if mask_jornal.any():

                total_jornal += (
                    df.loc[
                        mask_jornal,
                        'valor'
                    ].sum()
                )

                df = df[~mask_jornal]

            # ====================================================
            # PAGAMENTO
            # ====================================================
            mask_pagamento = (
                df[col_tipo]
                .astype(str)
                .str.upper()
                .str.contains(
                    'PAGAMENTO',
                    na=False
                )
            )

            if mask_pagamento.any():

                total_pagamento += abs(
                    df.loc[
                        mask_pagamento,
                        'valor'
                    ].sum()
                )

                df = df[~mask_pagamento]

        # ========================================================
        # VALORES NEGATIVOS
        # ========================================================
        mask_negativo = df['valor'] < 0

        if mask_negativo.any():

            total_negativo += abs(
                df.loc[
                    mask_negativo,
                    'valor'
                ].sum()
            )

            df = df[~mask_negativo]

        # ========================================================
        # SOMENTE VALORES POSITIVOS
        # ========================================================
        df = df[df['valor'] > 0]

        df = df.dropna(
            subset=['valor']
        )

        # ========================================================
        # PADRONIZAÇÃO DAS COLUNAS
        # ========================================================
        df['estabelecimento'] = (
            df[col_estab]
            if col_estab is not None
            else 'NÃO IDENTIFICADO'
        )

        df['tipo'] = (
            df[col_tipo]
            if col_tipo is not None
            else 'OUTROS'
        )

        df['categoria'] = df['tipo']

        # ========================================================
        # PARCELAS
        # ========================================================
        df[['parcela_atual', 'parcela_total']] = (

            df[col_parcela].apply(
                lambda x: pd.Series(
                    extrair_parcela(x)
                )
            )

            if col_parcela is not None

            else pd.Series(
                [(1, 1)] * len(df)
            ).apply(pd.Series)
        )

        df['parcela_total'] = (
            df['parcela_total']
            .fillna(1)
            .astype(int)
        )

        df['parcela_atual'] = (
            df['parcela_atual']
            .fillna(1)
            .astype(int)
        )

        lista_faturas.append(df)

        print(
            f"  ✅ {nome_arquivo_fatura}: "
            f"{len(df)} transações — "
            f"{formatar_valor_br(df['valor'].sum())}"
        )

    except Exception as e:

        print(
            f"  ⚠ Não foi possível carregar "
            f"{nome_arquivo_fatura}: "
            f"{str(e)[:100]}"
        )

        continue

# ============================================================
# IMPORTANTE: Registrar os cancelados no total, pois são gastos reais
# ============================================================
# Adicionar os cancelados ao total de gastos (já estão no df)
# total_cancelado já foi somado, mas os valores já estão no df

df_cartao_2026 = pd.concat(lista_faturas, ignore_index=True) if lista_faturas else pd.DataFrame()

# ============================================================
# DEDUPLICAÇÃO DE LANÇAMENTOS PROVAVELMENTE DUPLICADOS
# ============================================================
# O dedup anterior (mais abaixo, na seção de parcelas) só *avisava*
# sobre casos como "ULTRASSOM" e "ULTRASSOM MONK" (mesmo valor,
# mesma parcela, mesma data, nome de estabelecimento diferente) —
# nunca removia. Como o total_cartao é calculado ANTES daquela
# seção, os dois lançamentos continuavam sendo somados em dobro
# no total geral e no total mensal. Aqui removemos de fato,
# mantendo o registro com o nome mais descritivo (mais longo).
if len(df_cartao_2026):
    chave_dup = ['valor', 'parcela_atual', 'parcela_total', 'data_fatura']

    # IMPORTANTE: só aplicamos isso a compras PARCELADAS (parcela_total > 1).
    # Para compras à vista (1/1), valor repetido no mesmo mês é normalíssimo
    # (dois cafés de R$4, dois estacionamentos de R$20 etc.) e NÃO é duplicidade.
    mask_parcelada = df_cartao_2026['parcela_total'] > 1
    df_cartao_2026['_qtd_grupo'] = 0
    df_cartao_2026.loc[mask_parcelada, '_qtd_grupo'] = (
        df_cartao_2026.loc[mask_parcelada]
        .groupby(chave_dup)['valor']
        .transform('size')
    )
    duplicados = df_cartao_2026[mask_parcelada & (df_cartao_2026['_qtd_grupo'] > 1)]

    if len(duplicados):
        print(f"\n⚠ {len(duplicados)} lançamento(s) parcelado(s) com valor/parcela/data idênticos "
              f"e estabelecimento diferente — mantendo apenas 1 por grupo:")
        for _, row in duplicados.sort_values(chave_dup).iterrows():
            print(f"   • {row['estabelecimento']} | {formatar_valor_br(row['valor'])} | "
                  f"{row['parcela_atual']}/{row['parcela_total']} | "
                  f"{row['data_fatura'].strftime('%m/%Y')}")
        print("   ⚠ Confira se esses registros são realmente a mesma compra antes de confiar cegamente na remoção.")

    # Remove apenas dentro do subconjunto parcelado; compras 1/1 nunca são tocadas.
    parceladas_dedup = (
        df_cartao_2026[mask_parcelada]
        .assign(_len_nome=lambda d: d['estabelecimento'].astype(str).str.len())
        .sort_values('_len_nome', ascending=False)
        .drop_duplicates(subset=chave_dup, keep='first')
        .drop(columns=['_len_nome'])
    )
    df_cartao_2026 = (
        pd.concat([df_cartao_2026[~mask_parcelada], parceladas_dedup], ignore_index=True)
        .drop(columns=['_qtd_grupo'])
        .reset_index(drop=True)
    )

total_cartao = df_cartao_2026['valor'].sum() if len(df_cartao_2026) else 0
print(f"\n✅ TOTAL CARTÃO 2026 (após filtros): {len(df_cartao_2026)} transações — {formatar_valor_br(total_cartao)}")
print(f"   📌 Incluindo cancelados que ainda serão pagos: {formatar_valor_br(total_cancelado)}")


# ============================================================
# 2. CARREGAR DESPESAS DA CASA 2026
# ============================================================
print("\n" + "="*60)
print("🏠 CARREGANDO DESPESAS DA CASA 2026...")
print("="*60)

arquivos_casa = [f for f in arquivos_pasta_casa if '2026' in f and (f.endswith('.csv') or f.endswith('.gsheet') or f.endswith('.xlsx'))]
lista_casa = []

for arq in arquivos_casa:
    try:
        print(f"\n📄 Lendo: {arq}")
        df_c, origem = ler_despesa_casa(pasta_casa, arq)
        df_c.columns = df_c.columns.str.strip()

        match = re.search(r'(\d{4})-(\d{2})', arq)
        if not match:
            print(f"  ⚠ Pulando '{arq}': não foi possível extrair data do nome")
            continue

        ano_c, mes_c = int(match.group(1)), int(match.group(2))
        df_c['mes_ano'] = f"{mes_c:02d}/{ano_c}"
        df_c['ano'] = ano_c
        df_c['mes'] = mes_c

        if 'Valor' in df_c.columns:
            if pd.api.types.is_numeric_dtype(df_c['Valor']):
                # Excel já entrega número puro (ex.: 40.62) — não mexe.
                df_c['valor'] = pd.to_numeric(df_c['Valor'], errors='coerce')
            else:
                # Vem como texto, possivelmente formatado "R$ 1.234,56"
                df_c['valor'] = df_c['Valor'].astype(str).str.replace('R$', '', regex=False).str.strip()
                df_c['valor'] = df_c['valor'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
                df_c['valor'] = pd.to_numeric(df_c['valor'], errors='coerce')
        else:
            print(f"  ⚠ Coluna 'Valor' não encontrada em {arq}")
            continue

        df_c = df_c[df_c['valor'] > 0].dropna(subset=['valor'])

        if 'Categoria' in df_c.columns:
            df_c['categoria'] = df_c['Categoria']
        else:
            df_c['categoria'] = 'OUTROS'

        if 'Descrição' in df_c.columns:
            df_c['estabelecimento'] = df_c['Descrição']
        else:
            df_c['estabelecimento'] = 'NÃO IDENTIFICADO'

        df_c['tipo'] = df_c['categoria']
        lista_casa.append(df_c)
        print(f"  ✅ {arq}: {len(df_c)} despesas — {formatar_valor_br(df_c['valor'].sum())} [{origem}]")

    except Exception as e:
        print(f"  ⚠ Não foi possível carregar {arq}: {str(e)[:80]}")
        continue

df_casa = pd.concat(lista_casa, ignore_index=True) if lista_casa else pd.DataFrame()
total_casa = df_casa['valor'].sum() if len(df_casa) else 0
print(f"\n✅ TOTAL CASA 2026: {len(df_casa)} despesas — {formatar_valor_br(total_casa)}")

# ============================================================
# CATEGORIAS POR MÊS - RESUMO CONSOLE
# ============================================================
print("\n" + "="*60)
print("📊 CATEGORIAS POR MÊS (RESUMO):")
print("="*60)

df_total_mensal = pd.concat([
    df_cartao_2026[['mes_ano', 'categoria', 'valor']] if len(df_cartao_2026) else pd.DataFrame(),
    df_casa[['mes_ano', 'categoria', 'valor']] if len(df_casa) else pd.DataFrame()
], ignore_index=True)

nomes_pessoas = ['VINICIUS', 'EDNA', 'RY', 'FRANCISCO', 'MENEZES']
for nome in nomes_pessoas:
    df_total_mensal = df_total_mensal[~df_total_mensal['categoria'].astype(str).str.upper().str.contains(nome, na=False)]
df_total_mensal = df_total_mensal[df_total_mensal['categoria'].notna() & (df_total_mensal['categoria'] != '')]

categorias_por_mes = df_total_mensal.groupby(['mes_ano', 'categoria'])['valor'].sum().sort_values(ascending=False)
categorias_mensais = {}
for (mes_c, cat), val in categorias_por_mes.items():
    categorias_mensais.setdefault(mes_c, []).append((cat, val))

for mes in sorted(categorias_mensais.keys()):
    mes_num = int(mes.split('/')[0])
    mes_nome = nome_mes_pt(mes_num)

    print(f"\n📌 {mes_nome}/{mes.split('/')[1]}")
    print("-" * 40)

    top_categorias = sorted(categorias_mensais[mes], key=lambda x: x[1], reverse=True)[:5]

    for cat, valor in top_categorias:
        print(f"   {cat:<25} {formatar_valor_br(valor)}")

    total_mes = sum(v for _, v in categorias_mensais[mes])
    print(f"   {'Total do mês':<25} {formatar_valor_br(total_mes)}")
    print("-" * 40)

print("="*60)

# ============================================================
# 3. GASTOS MENSAIS CONSOLIDADOS
# ============================================================
gastos_cartao_mensal = df_cartao_2026.groupby('mes_ano')['valor'].sum() if len(df_cartao_2026) else pd.Series(dtype=float)
gastos_casa_mensal = df_casa.groupby('mes_ano')['valor'].sum() if len(df_casa) else pd.Series(dtype=float)

todos_meses = sorted(set(gastos_cartao_mensal.index) | set(gastos_casa_mensal.index))
gastos_mensais = pd.DataFrame(index=todos_meses, columns=['cartao', 'casa', 'total'], dtype=float)
for mes in todos_meses:
    c = gastos_cartao_mensal.get(mes, 0)
    h = gastos_casa_mensal.get(mes, 0)
    gastos_mensais.loc[mes] = [c, h, c + h]
gastos_mensais['acumulado_total'] = gastos_mensais['total'].cumsum()

# ============================================================
# 4. CALCULAR PARCELAS
# ============================================================
print("\n" + "="*60)
print("📦 CALCULANDO PARCELAS...")
print("="*60)

parceladas = (
    df_cartao_2026[df_cartao_2026['parcela_total'] > 1].copy()
    if len(df_cartao_2026)
    else pd.DataFrame()
)

df_parcelas_reais = pd.DataFrame()
df_parcelas_futuras = pd.DataFrame()

total_compras_parceladas = 0
total_parcelas_pagas = 0
valor_parcelas_pagas = 0
total_parcelas_futuras = 0
valor_parcelas_futuras = 0

parcelas_futuras_por_mes = {}

if len(parceladas) > 0:

    print(
        f"  🔍 Encontradas {len(parceladas)} "
        f"transações parceladas"
    )

    # ========================================================
    # 4.1 REMOVER SOMENTE DUPLICATAS EXATAMENTE IGUAIS
    # ========================================================

    colunas_duplicata_exata = [
        'estabelecimento',
        'valor',
        'parcela_atual',
        'parcela_total',
        'data_fatura'
    ]

    antes = len(parceladas)

    parceladas = parceladas.drop_duplicates(
        subset=colunas_duplicata_exata,
        keep='first'
    )

    depois = len(parceladas)

    print(
        f"  🔍 Duplicatas exatamente iguais removidas: "
        f"{antes - depois}"
    )

    print(
        f"  🔍 Transações após limpeza: "
        f"{depois}"
    )

    # ========================================================
    # 4.2 IDENTIFICAR POSSÍVEIS DUPLICIDADES
    # ========================================================

    # Não usamos estabelecimento nesta comparação.
    #
    # Exemplo:
    #
    # ULTRASSOM
    # ULTRASSOM MONK
    #
    # Mesmo valor + mesma parcela + mesmo total
    #
    # serão identificados para conferência.

    chave_possivel = [
        'valor',
        'parcela_atual',
        'parcela_total',
        'data_fatura'
    ]

    parceladas['_qtd_mesma_parcela'] = (
        parceladas
        .groupby(
            chave_possivel,
            dropna=False
        )['valor']
        .transform('size')
    )

    parceladas['possivel_duplicidade'] = (
        parceladas['_qtd_mesma_parcela'] > 1
    )

    qtd_possiveis = int(
        parceladas['possivel_duplicidade'].sum()
    )

    print(
        f"  ⚠ Possíveis duplicidades encontradas: "
        f"{qtd_possiveis}"
    )

    if qtd_possiveis > 0:

        print("\n  ⚠ REGISTROS PARA CONFERÊNCIA:")

        duplicadas = parceladas[
            parceladas['possivel_duplicidade']
        ].sort_values(
            chave_possivel
        )

        for _, row in duplicadas.iterrows():

            estabelecimento = str(
                row['estabelecimento']
            ).strip()

            valor = row['valor']

            parcela_atual = int(
                row['parcela_atual']
            )

            parcela_total = int(
                row['parcela_total']
            )

            data_fatura = row['data_fatura']

            print(
                f"    • {estabelecimento} | "
                f"{formatar_valor_br(valor)} | "
                f"{parcela_atual}/{parcela_total} | "
                f"{data_fatura.strftime('%m/%Y')}"
            )

    # ========================================================
    # 4.3 CRIAR SERIES_ID
    # ========================================================
    # Antes agrupávamos por (estabelecimento, parcela_total). Isso quebra
    # a MESMA compra em duas séries quando o nome do estabelecimento muda
    # de um mês para o outro (ex.: "ULTRASSOM" num mês e "ULTRASSOM MONK"
    # no outro) — daí uma parcela já paga numa série "fantasma" acabava
    # sendo projetada de novo como futura na outra. Agora agrupamos só por
    # parcela_total, e usamos proximidade de valor (em vez do nome) pra
    # decidir se uma parcela continua uma série já aberta — evita juntar
    # compras diferentes que só coincidem no número de parcelas.

    parceladas['series_id'] = -1

    series_counter = 0

    TOLERANCIA_VALOR = 0.15  # 15% de variação aceitável entre parcelas da mesma série

    for ptotal, grp in parceladas.groupby('parcela_total'):

        abertas = []

        for idx, row in grp.sort_values(
            'data_fatura'
        ).iterrows():

            p = row['parcela_atual']
            v = row['valor']

            if p == 1:

                series_counter += 1

                abertas.append({
                    'series_id': series_counter,
                    'last_p': 1,
                    'last_valor': v
                })

                parceladas.at[
                    idx,
                    'series_id'
                ] = series_counter

            else:

                candidatas = [
                    s for s in abertas
                    if s['last_p'] == p - 1
                ]

                # Prioriza candidatas com valor parecido (mesma compra
                # tende a ter parcelas de valor igual/muito próximo).
                candidatas_proximas = [
                    s for s in candidatas
                    if abs(s['last_valor'] - v) <= max(2.0, s['last_valor'] * TOLERANCIA_VALOR)
                ]

                candidatas_finais = candidatas_proximas if candidatas_proximas else candidatas

                if candidatas_finais:

                    melhor = min(
                        candidatas_finais,
                        key=lambda s:
                        abs(s['last_valor'] - v)
                    )

                    melhor['last_p'] = p
                    melhor['last_valor'] = v

                    parceladas.at[
                        idx,
                        'series_id'
                    ] = melhor['series_id']

                else:

                    series_counter += 1

                    abertas.append({
                        'series_id': series_counter,
                        'last_p': p,
                        'last_valor': v
                    })

                    parceladas.at[
                        idx,
                        'series_id'
                    ] = series_counter

    # ========================================================
    # 4.4 PREPARAR PARCELAS REAIS
    # ========================================================

    parceladas['status'] = 'provisorio'

    df_conhecidas = (
        parceladas
        .rename(
            columns={
                'parcela_atual': 'parcela_numero'
            }
        )[
            [
                'series_id',
                'estabelecimento',
                'tipo',
                'valor',
                'parcela_numero',
                'parcela_total',
                'mes_ano',
                'data_fatura',
                'ano',
                'mes',
                'status'
            ]
        ]
        .rename(
            columns={
                'valor': 'valor_parcela'
            }
        )
    )

    # ========================================================
    # 4.5 PROJETAR PARCELAS FUTURAS
    # ========================================================

    projetadas = []

    for sid, grp in df_conhecidas.groupby(
        'series_id'
    ):

        grp = grp.sort_values(
            'parcela_numero'
        )

        ultima = grp.iloc[-1]

        if (
            ultima['parcela_numero']
            < ultima['parcela_total']
        ):

            # Valor médio somente como referência
            # para projeção quando ainda não existe
            # a parcela futura na fatura.
            valor_serie = grp[
                'valor_parcela'
            ].mean()

            for p in range(
                int(ultima['parcela_numero']) + 1,
                int(ultima['parcela_total']) + 1
            ):

                data_proj = (
                    ultima['data_fatura']
                    + pd.DateOffset(
                        months=
                        p -
                        int(
                            ultima['parcela_numero']
                        )
                    )
                )

                projetadas.append({

                    'series_id':
                        sid,

                    'estabelecimento':
                        ultima['estabelecimento'],

                    'tipo':
                        ultima['tipo'],

                    'valor_parcela':
                        valor_serie,

                    'parcela_numero':
                        p,

                    'parcela_total':
                        ultima['parcela_total'],

                    'mes_ano':
                        data_proj.strftime('%m/%Y'),

                    'data_fatura':
                        data_proj,

                    'ano':
                        data_proj.year,

                    'mes':
                        data_proj.month,

                    'status':
                        'futura'
                })

    df_projetadas = pd.DataFrame(
        projetadas
    )

    # ========================================================
    # 4.6 NÃO PROJETAR PARCELA QUE JÁ EXISTE
    # ========================================================

    if len(df_projetadas) > 0:

        chave_conhecidas = set(
            zip(
                df_conhecidas[
                    'estabelecimento'
                ],
                df_conhecidas[
                    'parcela_numero'
                ],
                df_conhecidas[
                    'parcela_total'
                ]
            )
        )

        df_projetadas = (
            df_projetadas[
                ~df_projetadas.apply(
                    lambda r:
                    (
                        r['estabelecimento'],
                        r['parcela_numero'],
                        r['parcela_total']
                    )
                    in chave_conhecidas,
                    axis=1
                )
            ]
        )

    # ========================================================
    # 4.7 JUNTAR REAIS + FUTURAS
    # ========================================================

    if len(df_projetadas):

        df_todas = pd.concat(
            [
                df_conhecidas,
                df_projetadas
            ],
            ignore_index=True
        )

    else:

        df_todas = df_conhecidas.copy()

    # ========================================================
    # 4.8 IDENTIFICAR ÚLTIMA FATURA
    # ========================================================

    if nomes_base_2026:

        ultima_fatura = nomes_base_2026[-1]

        match = re.search(
            r'Fatura(\d{4})-(\d{2})-(\d{2})',
            ultima_fatura
        )

        if match:

            ano_ultima_fatura = int(
                match.group(1)
            )

            mes_ultima_fatura = int(
                match.group(2)
            )

            dia_ultima_fatura = int(
                match.group(3)
            )

            data_ultima_fatura = pd.Timestamp(
                ano_ultima_fatura,
                mes_ultima_fatura,
                dia_ultima_fatura
            )

        else:

            match = re.search(
                r'Fatura(\d{4})-(\d{2})',
                ultima_fatura
            )

            if match:

                ano_ultima_fatura = int(
                    match.group(1)
                )

                mes_ultima_fatura = int(
                    match.group(2)
                )

                data_ultima_fatura = pd.Timestamp(
                    ano_ultima_fatura,
                    mes_ultima_fatura,
                    1
                )

            else:

                data_ultima_fatura = pd.Timestamp.now()

    else:

        data_ultima_fatura = pd.Timestamp.now()

    print(
        f"  📅 Última fatura: "
        f"{data_ultima_fatura.strftime('%m/%Y')}"
    )

    # ========================================================
    # 4.9 CLASSIFICAR PAGAS X FUTURAS
    # ========================================================

    df_todas['status'] = np.where(
        df_todas['data_fatura']
        < data_ultima_fatura,
        'paga',
        'futura'
    )

    df_pagas = df_todas[
        df_todas['status'] == 'paga'
    ].copy()

    df_futuras = df_todas[
        df_todas['status'] == 'futura'
    ].copy()

    # ========================================================
    # 4.10 REMOVER DAS FUTURAS O QUE JÁ FOI PAGO
    # ========================================================

    if len(df_pagas) > 0:

        ultimas_pagas = (
            df_pagas
            .groupby('series_id')
            .agg({
                'parcela_numero': 'max'
            })
            .reset_index()
        )

        ultima_paga_dict = dict(
            zip(
                ultimas_pagas['series_id'],
                ultimas_pagas['parcela_numero']
            )
        )

        if len(df_futuras) > 0:

            df_futuras_filtradas = []

            for _, row in df_futuras.iterrows():

                sid = row['series_id']

                parcela_num = (
                    row['parcela_numero']
                )

                if sid in ultima_paga_dict:

                    ultimo_pago = (
                        ultima_paga_dict[sid]
                    )

                    if parcela_num > ultimo_pago:

                        df_futuras_filtradas.append(
                            row
                        )

                else:

                    df_futuras_filtradas.append(
                        row
                    )

            if df_futuras_filtradas:

                df_futuras = pd.DataFrame(
                    df_futuras_filtradas
                )

                print(
                    f"  🔍 Parcelas futuras após "
                    f"remover já pagas: "
                    f"{len(df_futuras)}"
                )

            else:

                df_futuras = pd.DataFrame()

    else:

        df_futuras = df_todas[
            df_todas['status'] == 'futura'
        ].copy()

    # ========================================================
    # 4.11 RESULTADOS FINAIS
    # ========================================================

    df_parcelas_reais = df_pagas.copy()

    df_parcelas_futuras = df_futuras.copy()

    total_compras_parceladas = (
        df_todas['series_id'].nunique()
    )

    total_parcelas_pagas = len(
        df_parcelas_reais
    )

    valor_parcelas_pagas = (
        df_parcelas_reais[
            'valor_parcela'
        ].sum()
        if len(df_parcelas_reais)
        else 0
    )

    total_parcelas_futuras = len(
        df_parcelas_futuras
    )

    valor_parcelas_futuras = (
        df_parcelas_futuras[
            'valor_parcela'
        ].sum()
        if len(df_parcelas_futuras)
        else 0
    )

    parcelas_futuras_por_mes = (
        {
            m: g
            for m, g
            in df_parcelas_futuras.groupby(
                'mes_ano'
            )
        }
        if len(df_parcelas_futuras)
        else {}
    )

    # ========================================================
    # 4.12 LIMPAR COLUNAS AUXILIARES
    # ========================================================

    for df_temp in [
        df_parcelas_reais,
        df_parcelas_futuras
    ]:

        if len(df_temp):

            df_temp.drop(
                columns=[
                    '_qtd_mesma_parcela',
                    'possivel_duplicidade'
                ],
                errors='ignore',
                inplace=True
            )


print(
    f"  📦 Compras parceladas: "
    f"{total_compras_parceladas}"
)

print(
    f"  ✅ Parcelas pagas: "
    f"{total_parcelas_pagas} — "
    f"{formatar_valor_br(valor_parcelas_pagas)}"
)

print(
    f"  ⏳ Parcelas futuras: "
    f"{total_parcelas_futuras} — "
    f"{formatar_valor_br(valor_parcelas_futuras)}"
)

# ============================================================
# 5. GERAR RELATÓRIO HTML 2026
# ============================================================
print("\n" + "="*60)
print("📄 GERANDO RELATÓRIO HTML 2026...")
print("="*60)

# HEADER
data_hora_atual = agora_brasilia()
data_formatada = data_hora_atual.strftime('%d/%m/%Y')
hora_formatada = data_hora_atual.strftime('%H:%M')
mes_atual = data_hora_atual.month
mes_atual_nome = nome_mes_pt(mes_atual)

# ============================================================
# IDENTIFICAR A FATURA VIGENTE (última fatura carregada)
# ============================================================
if nomes_base_2026:
    ultima_fatura = nomes_base_2026[-1]
    # Extrair mês da última fatura
    match = re.search(r'Fatura(\d{4})-(\d{2})-(\d{2})', ultima_fatura)
    if match:
        mes_ultima_fatura = int(match.group(2))
        ano_ultima_fatura = int(match.group(1))
    else:
        match = re.search(r'Fatura(\d{4})-(\d{2})', ultima_fatura)
        if match:
            mes_ultima_fatura = int(match.group(2))
            ano_ultima_fatura = int(match.group(1))
        else:
            mes_ultima_fatura = mes_atual
            ano_ultima_fatura = data_hora_atual.year

    mes_fatura_nome = nome_mes_pt(mes_ultima_fatura)
    fatura_vigente_texto = f"{mes_fatura_nome}/{ano_ultima_fatura}"
    fatura_vigente_detalhe = ultima_fatura
else:
    fatura_vigente_texto = "Nenhuma fatura carregada"
    fatura_vigente_detalhe = "N/A"

df_total_mensal = pd.concat([
    df_cartao_2026[['mes_ano', 'categoria', 'valor']] if len(df_cartao_2026) else pd.DataFrame(),
    df_casa[['mes_ano', 'categoria', 'valor']] if len(df_casa) else pd.DataFrame()
], ignore_index=True)

nomes_pessoas = ['VINICIUS', 'EDNA']
for nome in nomes_pessoas:
    df_total_mensal = df_total_mensal[~df_total_mensal['categoria'].astype(str).str.upper().str.contains(nome, na=False)]
df_total_mensal = df_total_mensal[df_total_mensal['categoria'].notna() & (df_total_mensal['categoria'] != '')]

categorias_por_mes = df_total_mensal.groupby(['mes_ano', 'categoria'])['valor'].sum().sort_values(ascending=False)
categorias_mensais = {}
for (mes_c, cat), val in categorias_por_mes.items():
    categorias_mensais.setdefault(mes_c, []).append((cat, val))

total_geral = gastos_mensais["total"].sum()
print(f"\n💰 TOTAL GERAL 2026: {formatar_valor_br(total_geral)}")

gastos_mensais_2026 = gastos_mensais[gastos_mensais.index.str.endswith('/2026')] if len(gastos_mensais) > 0 else pd.DataFrame()
meses_realizados = list(gastos_mensais_2026.index)
gastos_mensais_realizados = gastos_mensais_2026.loc[meses_realizados] if meses_realizados else pd.DataFrame()


gastos_mensais_2026 = gastos_mensais_realizados.copy()

total_bruto_cartao = total_bruto_real if 'total_bruto_real' in locals() else (df_cartao_2026['valor'].sum() if len(df_cartao_2026) > 0 else 0)
valores_positivos = df_cartao_2026[df_cartao_2026['valor'] > 0]['valor'].sum() if len(df_cartao_2026) > 0 else 0
total_faturas_carregadas = len(nomes_base_2026) if 'nomes_base_2026' in locals() else 0

# ============================================================
# TOTAIS "REALIZADOS" USADOS NO HTML (faltavam no notebook original)
# ============================================================
# total_cartao já é o total líquido de verdade (soma de df_cartao_2026,
# calculada logo após a deduplicação, já sem jornal/pagamento/negativo).
# Antes eu estava subtraindo total_excluido (que soma valores ABSOLUTOS)
# de um total que já estava líquido — isso descontava a mesma coisa
# duas vezes e jogava o resultado para negativo. Corrigido abaixo:
total_cartao_realizado = total_cartao

# total_excluido aqui é só para exibição ("quanto foi tirado da conta"),
# calculado pela diferença real entre bruto e líquido — sempre consistente,
# mesmo quando jornal/pagamento/negativo têm sinais diferentes entre si.
total_excluido = total_bruto_cartao - total_cartao_realizado

total_casa_realizado = gastos_mensais_2026['casa'].sum() if len(gastos_mensais_2026) > 0 else 0
total_geral_realizado = total_cartao_realizado + total_casa_realizado


meses_considerados_lista = []
for mes in sorted(gastos_mensais_2026.index):
    mes_num = int(mes.split('/')[0])
    meses_considerados_lista.append(nome_mes_pt(mes_num))
meses_considerados = ', '.join(meses_considerados_lista) if meses_considerados_lista else 'Nenhum'

total_parcelas_futuras_2026 = df_parcelas_futuras['valor_parcela'].sum() if len(df_parcelas_futuras) > 0 else 0
total_parcelas_futuras_count_2026 = len(df_parcelas_futuras) if len(df_parcelas_futuras) > 0 else 0
total_parcelas_pagas_2026 = df_parcelas_reais['valor_parcela'].sum() if len(df_parcelas_reais) > 0 else 0
total_parcelas_pagas_count_2026 = len(df_parcelas_reais) if len(df_parcelas_reais) > 0 else 0

cards_mensais = ""
for mes in gastos_mensais_2026.index:
    cartao = gastos_mensais_2026.loc[mes, 'cartao']
    casa = gastos_mensais_2026.loc[mes, 'casa']
    total = gastos_mensais_2026.loc[mes, 'total']
    acumulado = gastos_mensais_2026.loc[mes, 'acumulado_total']

    mes_num = int(mes.split('/')[0])
    mes_nome = nome_mes_pt(mes_num)

    cards_mensais += f"""
        <div class="card-mes">
            <div class="card-mes-header">
                <span class="card-mes-titulo">📌 {mes_nome}/{mes.split('/')[1]}</span>
                <span class="card-mes-total">💰 {formatar_valor_br(total)}</span>
            </div>
            <div class="card-mes-detalhes">
                <div class="card-mes-item"><span class="card-mes-label">💳 Cartão</span><span class="card-mes-valor">{formatar_valor_br(cartao)}</span></div>
                <div class="card-mes-item"><span class="card-mes-label">🏠 Casa</span><span class="card-mes-valor">{formatar_valor_br(casa)}</span></div>
                <div class="card-mes-item"><span class="card-mes-label">📈 Acumulado</span><span class="card-mes-valor" style="color:#2E86AB;font-weight:bold;">{formatar_valor_br(acumulado)}</span></div>
            </div>
        </div>
    """
if cards_mensais == "":
    cards_mensais = '<div class="info-box">⚠ Nenhum gasto mensal de 2026 encontrado</div>'

html_content = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Relatório Financeiro 2026</title>
<style>
* {{ margin:0; padding:0; box-sizing:border-box; }}
body {{ font-family:'Segoe UI',Tahoma,Geneva,Verdana,sans-serif; background:#f0f2f5; min-height:100vh; padding:20px; }}
.container {{ max-width:1400px; margin:0 auto; background:white; padding:30px; border-radius:16px; box-shadow:0 10px 40px rgba(0,0,0,0.08); }}
h1 {{ color:#1a1a2e; border-bottom:3px solid #2E86AB; padding-bottom:15px; margin-bottom:30px; font-size:28px; display:flex; align-items:center; gap:10px; }}
h1::before {{ content:"📊"; }}
h2 {{ color:#1a1a2e; margin:30px 0 20px 0; padding:10px 15px; background:#f8f9fa; border-left:4px solid #2E86AB; border-radius:4px; font-size:20px; }}
h3 {{ color:#2E86AB; margin:10px 0 8px 0; font-size:16px; }}
.header-info {{ background:linear-gradient(135deg,#f5f7fa 0%,#c3cfe2 100%); padding:15px 20px; border-radius:10px; margin-bottom:30px; display:flex; justify-content:space-between; flex-wrap:wrap; gap:15px; }}
.header-info span {{ font-size:14px; color:#555; }}
.header-info strong {{ color:#2E86AB; }}
.metricas {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:20px; margin:25px 0; }}
.metrica {{ padding:20px; border-radius:12px; text-align:center; box-shadow:0 2px 10px rgba(0,0,0,0.06); }}
.metrica .valor {{ font-size:26px; font-weight:bold; margin-top:6px; font-family:'Courier New',monospace; }}
.metrica .label {{ font-size:13px; opacity:0.8; text-transform:uppercase; letter-spacing:0.5px; }}
.metrica-verde {{ background:#e8f5e9; color:#1a5a3a; }}
.metrica-azul {{ background:#e3f2fd; color:#1a3a5a; }}
.metrica-rosa {{ background:#fce4ec; color:#5a1a2a; }}
.metrica-roxo {{ background:#f3e5f5; color:#3a1a4a; }}
.metrica-cinza {{ background:#f5f5f5; color:#555; }}
.grid-2 {{ display:grid; grid-template-columns:1fr 1fr; gap:25px; margin:25px 0; }}
@media (max-width:768px) {{ .grid-2 {{ grid-template-columns:1fr; }} }}
.info-box {{ background:#f8f9fa; padding:18px 22px; border-radius:10px; margin:15px 0; border-left:4px solid #2E86AB; }}
.info-box strong {{ color:#2E86AB; }}
.footer {{ text-align:center; margin-top:40px; padding:20px; background:#f8f9fa; border-radius:10px; color:#7f8c8d; font-size:13px; }}
.footer strong {{ color:#2E86AB; }}
.insights {{ background:#e8f5e9; padding:15px 20px; border-radius:10px; margin:15px 0; border-left:4px solid #27ae60; }}
.insights li {{ margin:6px 0; list-style:none; }}
.insights li::before {{ content:"💡 "; }}
.cards-mensais {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:16px; margin:20px 0; }}
.card-mes {{ background:white; border-radius:12px; padding:16px 18px; box-shadow:0 2px 8px rgba(0,0,0,0.06); border:1px solid #e9ecef; }}
.card-mes-header {{ display:flex; justify-content:space-between; align-items:center; margin-bottom:10px; padding-bottom:8px; border-bottom:2px solid #f0f2f5; }}
.card-mes-titulo {{ font-weight:600; font-size:15px; color:#1a1a2e; }}
.card-mes-total {{ font-weight:700; font-size:16px; color:#2E86AB; }}
.card-mes-detalhes {{ display:flex; flex-direction:column; gap:4px; }}
.card-mes-item {{ display:flex; justify-content:space-between; font-size:13px; padding:2px 0; }}
.card-mes-label {{ color:#7f8c8d; }}
.card-mes-valor {{ font-weight:600; color:#1a1a2e; }}
.categorias-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(250px,1fr)); gap:12px; margin:15px 0; }}
.categoria-card {{ background:#f8f9fa; padding:10px 14px; border-radius:8px; border-left:3px solid #2E86AB; display:flex; justify-content:space-between; align-items:center; }}
.categoria-card .nome {{ font-weight:500; font-size:13px; }}
.categoria-card .valor {{ font-weight:600; font-size:13px; color:#27ae60; }}
.parcelas-mes {{ margin-bottom:20px; background:#f8f9fa; border-radius:10px; padding:15px 18px; border-left:4px solid #2E86AB; }}
.parcelas-mes h3 {{ color:#2E86AB; margin-bottom:10px; font-size:16px; }}
.parcela-item {{ display:flex; justify-content:space-between; padding:4px 0; font-size:13px; border-bottom:1px solid #eee; }}
.parcela-item:last-child {{ border-bottom:none; }}
.parcela-item .estab {{ font-weight:500; }}
.parcela-item .valor {{ font-weight:600; color:#27ae60; }}
.parcela-item .parcela-info {{ color:#7f8c8d; font-size:12px; }}
.legenda {{ background:#fef9e7; padding:18px 22px; border-radius:10px; margin:20px 0; border-left:4px solid #f39c12; }}
.legenda-item {{ padding:4px 0; font-size:13px; display:flex; justify-content:space-between; border-bottom:1px solid #fdebd0; }}
.legenda-item:last-child {{ border-bottom:none; }}
.legenda-item .label {{ color:#555; }}
.legenda-item .valor {{ font-weight:600; color:#1a1a2e; }}
.legenda-item .excluido {{ color:#e74c3c; }}
.legenda-item .incluido {{ color:#27ae60; }}
.legenda-item .positivo {{ color:#2980b9; }}
.legenda-item .cancelado {{ color:#e67e22; }}
</style>
</head>
<body>
<div class="container">
    <h1>RELATÓRIO FINANCEIRO 2026</h1>

    <!-- ============================================================ -->
    <!-- MOSTRAR A FATURA VIGENTE -->
    <!-- ============================================================ -->
    <div style="background: #e8f4fd; padding: 15px 20px; border-radius: 10px; margin-bottom: 20px; border-left: 4px solid #2E86AB;">
        <p style="font-size: 16px; color: #1a1a2e; margin: 0;">
            📌 <strong>Fatura vigente:</strong> {fatura_vigente_texto}
            <span style="font-size: 14px; color: #555; margin-left: 15px;">
                (Arquivo: {fatura_vigente_detalhe})
            </span>
        </p>
        <p style="font-size: 13px; color: #666; margin: 5px 0 0 0;">
            📅 Data atual: {data_formatada} às {hora_formatada}
        </p>
    </div>

    <div class="header-info">
        <span>📅 <strong>Gerado em:</strong> {data_formatada} às {hora_formatada}</span>
        <span>📊 <strong>Ano:</strong> 2026</span>
        <span>📁 <strong>Arquivos processados:</strong> {len(arquivos)}</span>
    </div>

    <div class="metricas">
        <div class="metrica metrica-verde"><div class="label">💰 Total Gasto 2026</div><div class="valor">{formatar_valor_br(total_geral_realizado)}</div></div>
        <div class="metrica metrica-azul"><div class="label">💳 Cartão 2026</div><div class="valor">{formatar_valor_br(total_cartao_realizado)}</div></div>
        <div class="metrica metrica-rosa"><div class="label">🏠 Despesas Casa 2026</div><div class="valor">{formatar_valor_br(total_casa_realizado)}</div></div>
        <div class="metrica metrica-roxo"><div class="label">🔮 Parcelas Futuras 2026</div><div class="valor">{formatar_valor_br(total_parcelas_futuras_2026)}</div></div>
    </div>

    <h2>💳 RESUMO DE PARCELAS 2026</h2>
    <div class="grid-2">
        <div class="info-box" style="border-left-color:#27ae60;background:#e8f5e9;">
            <h3 style="color:#27ae60;">✅ Parcelas Já Cobradas 2026</h3>
            <p><strong>Quantidade:</strong> {total_parcelas_pagas_count_2026}</p>
            <p><strong>Valor total:</strong> {formatar_valor_br(total_parcelas_pagas_2026)}</p>
            <p><strong>Compras parceladas distintas:</strong> {total_compras_parceladas}</p>
        </div>
        <div class="info-box" style="border-left-color:#e74c3c;background:#fde8e8;">
            <h3 style="color:#e74c3c;">⏳ Parcelas Futuras 2026</h3>
            <p><strong>Quantidade:</strong> {total_parcelas_futuras_count_2026}</p>
            <p><strong>Valor total:</strong> {formatar_valor_br(total_parcelas_futuras_2026)}</p>
            <p><strong>Média por parcela:</strong> {formatar_valor_br(total_parcelas_futuras_2026 / total_parcelas_futuras_count_2026 if total_parcelas_futuras_count_2026 > 0 else 0)}</p>
        </div>
    </div>

    <h2>📊 GASTOS MENSAIS CONSOLIDADOS 2026</h2>
    <div class="cards-mensais">{cards_mensais}</div>

    <h2>📊 CATEGORIAS POR MÊS 2026</h2>
"""

if len(categorias_mensais) > 0:
    categorias_mensais_2026 = {mes: cats for mes, cats in categorias_mensais.items() if mes.endswith('/2026')}
    for mes in sorted(categorias_mensais_2026.keys()):
        mes_num = int(mes.split('/')[0])
        mes_nome = nome_mes_pt(mes_num)
        categorias_ordenadas = sorted(categorias_mensais_2026[mes], key=lambda x: x[1], reverse=True)
        html_content += f"""
    <div style="margin-bottom:16px;background:#f8f9fa;border-radius:10px;padding:14px 18px;border-left:4px solid #2E86AB;">
        <h3 style="margin-bottom:10px;color:#2E86AB;font-size:16px;">📌 {mes_nome}/{mes.split('/')[1]}</h3>
        <div class="categorias-grid">
        """
        for categoria, valor in categorias_ordenadas[:8]:
            html_content += f"""
            <div class="categoria-card"><span class="nome">{categoria}</span><span class="valor">{formatar_valor_br(valor)}</span></div>
            """
        if len(categorias_ordenadas) > 8:
            html_content += f"""
            <div class="categoria-card" style="border-left-color:#95a5a6;background:#ecf0f1;"><span class="nome">+ {len(categorias_ordenadas)-8} outras</span><span class="valor">{formatar_valor_br(sum(v for _,v in categorias_ordenadas[8:]))}</span></div>
            """
        html_content += "</div></div>"
else:
    html_content += '<div class="info-box">⚠ Nenhuma categoria encontrada para 2026</div>'

# ============================================================
# PARCELAS FUTURAS - HTML (TODAS AS PARCELAS PENDENTES)
# ============================================================
html_content += "<h2>🔮 PARCELAS FUTURAS - DETALHADAS POR MÊS</h2>"

# Determinar o mês atual
mes_atual_num = datetime.now().month
ano_atual = datetime.now().year

# Determinar o mês seguinte
mes_seguinte = mes_atual_num + 1 if mes_atual_num < 12 else 1
ano_seguinte = ano_atual if mes_atual_num < 12 else ano_atual + 1

print(f"\n📌 Mostrando parcelas a partir de {nome_mes_pt(mes_seguinte)}/{ano_seguinte}")

# Filtrar parcelas futuras - mostrar a partir do mês seguinte
parcelas_futuras_filtradas = {}
if len(parcelas_futuras_por_mes) > 0:
    for mes, grupo in parcelas_futuras_por_mes.items():
        mes_num = int(mes.split('/')[0])
        ano_mes = int(mes.split('/')[1])

        # ============================================================
        # CORREÇÃO: Mostrar a partir do mês seguinte (incluindo)
        # ============================================================
        if ano_mes > ano_seguinte or (ano_mes == ano_seguinte and mes_num >= mes_seguinte):
            parcelas_futuras_filtradas[mes] = grupo
            print(f"  ✅ Incluindo: {nome_mes_pt(mes_num)}/{ano_mes}")
        else:
            print(f"  ⏭️ Pulando: {nome_mes_pt(mes_num)}/{ano_mes}")

# Se não houver parcelas filtradas, mostrar todas as futuras
if len(parcelas_futuras_filtradas) == 0 and len(parcelas_futuras_por_mes) > 0:
    parcelas_futuras_filtradas = parcelas_futuras_por_mes
    print("  📌 Nenhuma parcela futura encontrada a partir do mês seguinte, mostrando todas")


# Se não houver parcelas filtradas, mostrar todas as futuras
if len(parcelas_futuras_filtradas) == 0 and len(parcelas_futuras_por_mes) > 0:
    # Se não encontrou nada a partir do mês seguinte, mostra todas
    parcelas_futuras_filtradas = parcelas_futuras_por_mes
    print("  📌 Nenhuma parcela futura encontrada a partir do mês seguinte, mostrando todas")

if len(parcelas_futuras_filtradas) > 0:
    # Ordenar por ano e mês
    for mes in sorted(parcelas_futuras_filtradas.keys(), key=lambda m: (int(m.split('/')[1]), int(m.split('/')[0]))):
        mes_num = int(mes.split('/')[0])
        ano_mes = int(mes.split('/')[1])
        mes_nome = nome_mes_pt(mes_num)
        grupo = parcelas_futuras_filtradas[mes]
        total_mes = grupo['valor_parcela'].sum()

        html_content += f"""
    <div class="parcelas-mes">
        <h3>📌 {mes_nome}/{ano_mes} - Total: {formatar_valor_br(total_mes)}</h3>
        """
        for _, row in grupo.sort_values('valor_parcela', ascending=False).iterrows():
            html_content += f"""
        <div class="parcela-item">
            <span class="estab">{str(row['estabelecimento'])[:50]}</span>
            <span><span class="valor">{formatar_valor_br(row['valor_parcela'])}</span> <span class="parcela-info">(Parcela {row['parcela_numero']}/{row['parcela_total']})</span></span>
        </div>
        """
        html_content += "</div>"
else:
    html_content += '<div class="info-box">✅ Nenhuma parcela futura identificada</div>'

html_content += "<h2>💡 INSIGHTS AUTOMÁTICOS - 2026</h2><div class=\"insights\"><ul>"

insights = []
if total_geral_realizado > 0:
    if total_casa_realizado > 0:
        insights.append(f"🏠 Despesas da casa representam {(total_casa_realizado/total_geral_realizado*100):.1f}% do total gasto em 2026")
    if total_cartao_realizado > 0:
        insights.append(f"💳 Gastos do cartão representam {(total_cartao_realizado/total_geral_realizado*100):.1f}% do total gasto em 2026")
    if total_parcelas_futuras_2026 > 0:
        insights.append(f"🔮 Parcelas futuras: {formatar_valor_br(total_parcelas_futuras_2026)} a vencer em 2026")
    if total_parcelas_pagas_count_2026 > 0:
        insights.append(f"✅ Parcelas já cobradas: {total_parcelas_pagas_count_2026} ({formatar_valor_br(total_parcelas_pagas_2026)})")
    if len(gastos_mensais_2026) > 0:
        gastos_com_valor = gastos_mensais_2026[gastos_mensais_2026['total'] > 0]
        if len(gastos_com_valor) > 0:
            mes_maior = gastos_com_valor['total'].idxmax()
            mes_num = int(mes_maior.split('/')[0])
            mes_nome = nome_mes_pt(mes_num)
            insights.append(f"📈 O mês com maior gasto em 2026 foi {mes_nome}: {formatar_valor_br(gastos_com_valor.loc[mes_maior,'total'])}")
            if len(gastos_com_valor) > 1:
                mes_menor = gastos_com_valor['total'].idxmin()
                mes_num = int(mes_menor.split('/')[0])
                mes_nome = nome_mes_pt(mes_num)
                insights.append(f"📉 O mês com menor gasto em 2026 foi {mes_nome}: {formatar_valor_br(gastos_com_valor.loc[mes_menor,'total'])}")
if len(insights) == 0:
    insights.append("📊 Análise em andamento...")
for insight in insights:
    html_content += f"<li>{insight}</li>\n"

html_content += f"""
    </ul></div>

    <h2>📋 LEGENDA - ENTENDENDO OS VALORES</h2>
    <div class="legenda">
        <h3 style="color:#f39c12;margin-bottom:12px;">📌 O que foi considerado neste relatório:</h3>

        <div class="legenda-item">
            <span class="label">✅ Total de faturas carregadas:</span>
            <span class="valor">{total_faturas_carregadas} faturas</span>
        </div>

        <div class="legenda-item">
            <span class="label">✅ Meses considerados (realizados):</span>
            <span class="valor">{meses_considerados}</span>
        </div>

        <div class="legenda-item" style="background:#d6eaf8;border-radius:4px;padding:4px 8px;margin:4px 0;">
            <span class="label">➕ Valores POSITIVOS (gastos reais):</span>
            <span class="valor positivo">{formatar_valor_br(valores_positivos)}</span>
        </div>

        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#2980b9;">
            <span class="label">  • O que são:</span>
            <span style="color:#555;font-weight:normal;">Todas as compras, serviços e despesas com valor positivo</span>
        </div>

        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#2980b9;">
            <span class="label">  • Como são tratados:</span>
            <span style="color:#555;font-weight:normal;">✅ INCLUÍDOS no total de gastos</span>
        </div>

        <div class="legenda-item" style="background:#fde8e8;border-radius:4px;padding:4px 8px;margin:8px 0;">
            <span class="label">🚫 Valores EXCLUÍDOS do cálculo:</span>
            <span class="valor excluido">- {formatar_valor_br(total_excluido)}</span>
        </div>
"""

# Reembolsos: aviso informativo, sem alterar o valor das faturas.
resumo_reembolsos = {}
if len(df_cartao_2026) and 'categoria' in df_cartao_2026.columns:
    categorias = df_cartao_2026['categoria'].fillna('').astype(str).str.strip().str.upper()
    for categoria_reembolso in ('VIAGEM TRABALHO REEMBOLSO', 'TRABALHO REEMBOLSO'):
        mask = categorias.eq(categoria_reembolso)
        resumo_reembolsos[categoria_reembolso] = (float(df_cartao_2026.loc[mask, 'valor'].sum()), int(mask.sum()))

for categoria_reembolso, (valor_reembolso, quantidade) in resumo_reembolsos.items():
    if quantidade:
        html_content += f"""
        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#2E86AB;">
            <span class="label">• {categoria_reembolso} (a reembolsar pela empresa — incluído no cartão):</span>
            <span class="valor positivo">{formatar_valor_br(valor_reembolso)}</span>
        </div>
        """

if total_cancelado > 0:
    html_content += f"""
        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#e67e22;">
            <span class="label">  • ❌ CANCELADO (estornos de compras):</span>
            <span class="valor cancelado">{formatar_valor_br(total_cancelado)}</span>
        </div>
        <div class="legenda-item" style="padding-left:40px;font-size:11px;color:#7f8c8d;">
            <span class="label">    Transações:</span>
            <span style="color:#555;font-weight:normal;">{', '.join([str(t)[:30] for t in transacoes_canceladas[:3]])}{'...' if len(transacoes_canceladas) > 3 else ''}</span>
        </div>
    """

if total_jornal > 0:
    html_content += f"""
        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#e74c3c;">
            <span class="label">  • 📰 JORNAL (despesas de trabalho):</span>
            <span class="valor excluido">{formatar_valor_br(total_jornal)}</span>
        </div>
    """

if total_pagamento > 0:
    html_content += f"""
        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#e74c3c;">
            <span class="label">  • 💳 PAGAMENTO (transferências):</span>
            <span class="valor excluido">{formatar_valor_br(total_pagamento)}</span>
        </div>
    """

if total_negativo > 0:
    html_content += f"""
        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#e74c3c;">
            <span class="label">  • ➖ Valores negativos:</span>
            <span class="valor excluido">{formatar_valor_br(total_negativo)}</span>
        </div>
    """

html_content += f"""
        <div class="legenda-item" style="padding-left:20px;font-size:12px;color:#e74c3c;">
            <span class="label">  • Como são tratados:</span>
            <span style="color:#555;font-weight:normal;">🚫 EXCLUÍDOS para não distorcer o total de gastos</span>
        </div>

        <div class="legenda-item" style="background:#e8f5e9;border-radius:4px;padding:4px 8px;margin:8px 0;border-top:2px solid #27ae60;">
            <span class="label">✅ Total CONSIDERADO no relatório:</span>
            <span class="valor incluido">{formatar_valor_br(total_cartao_realizado)}</span>
        </div>

        <div style="margin-top:15px;padding-top:10px;border-top:1px solid #fdebd0;">
            <p style="font-size:12px;color:#7f8c8d;line-height:1.8;">
                <strong>🔍 Explicação detalhada dos filtros aplicados:</strong><br>
                • <strong>Valores positivos</strong> → são <strong>incluídos</strong> no total (representam gastos reais)<br>
                • <strong>VIAGEM TRABALHO REEMBOLSO / TRABALHO REEMBOLSO</strong> → permanecem no cartão; reembolso previsto pela empresa<br>
                • <strong>CANCELADO</strong> → são <strong>excluídos</strong> (compras canceladas/estornadas que não geraram gasto efetivo)<br>
                • <strong>JORNAL</strong> → são <strong>excluídos</strong> (despesas de trabalho que não são gastos pessoais)<br>
                • <strong>PAGAMENTO</strong> → são <strong>excluídos</strong> (transferências entre contas, não são gastos)<br>
                • <strong>Valores negativos</strong> → são <strong>excluídos</strong> (pagamentos de fatura que reduzem o saldo)<br>
                • <strong>Meses futuros</strong> (após {mes_atual_nome}) → são <strong>excluídos</strong> do total, mas aparecem nas projeções
            </p>
        </div>
    </div>

    <div class="footer">
        <p>📅 Relatório gerado automaticamente em <strong>{data_formatada} às {hora_formatada}</strong></p>
        <p>📊 Dados consolidados de cartão de crédito e despesas da casa - <strong>ANO 2026</strong></p>
        <p>📊 Período considerado: <strong>Janeiro a {mes_atual_nome} (meses realizados)</strong></p>
        <p>💡 Apenas gastos reais (valores positivos) são considerados</p>
        <p style="color:#95a5a6;font-size:11px;margin-top:5px;">✅ Cada parcela usa o valor real cobrado na fatura — sem recálculo nem duplicação</p>
        <p style="color:#95a5a6;font-size:10px;margin-top:8px;border-top:1px solid #eee;padding-top:8px;">
            📌 <strong>Para conferência:</strong> Total bruto R$ {formatar_valor_br(total_bruto_cartao)} - Excluídos R$ {formatar_valor_br(total_excluido)} = Total líquido R$ {formatar_valor_br(total_cartao_realizado)}
        </p>
    </div>
</div>
</body>
</html>
"""

nome_arquivo = f"relatorio_completo_2026_{data_hora_atual.strftime('%Y%m%d_%H%M')}.html"
caminho_html = os.path.join(pasta_relatorios, nome_arquivo)
with open(caminho_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n✅ Relatório HTML 2026 salvo em: {caminho_html}")

from google.colab import files
files.download(caminho_html)
print("⬇️  Download do relatório 2026 iniciado")


# ============================================================
# 6. CARREGAR DADOS DE 2025 PARA COMPARAÇÃO
# ============================================================
print("\n" + "="*60)
print("📊 CARREGANDO DADOS DE 2025 PARA COMPARAÇÃO...")
print("="*60)

faturas_2025 = sorted(faturas_2025)

lista_faturas_2025 = []

for nome_arquivo_fatura in faturas_2025:

    try:

        df = df_faturas_origem[
            df_faturas_origem["Source.Name"].astype(str).str.strip()
            == nome_arquivo_fatura
        ].copy()

        print(
            f"\n📄 Processando: "
            f"{nome_arquivo_fatura}  [Faturas.xlsx]"
        )

        if df.empty:
            print("  ⚠ Nenhum registro encontrado.")
            continue

        df.columns = df.columns.str.strip()

        col_estab = achar_coluna(
            df,
            ['estabelecimento', 'descrição', 'descricao']
        )

        col_tipo = achar_coluna(
            df,
            ['tipo', 'categoria']
        )

        col_valor = achar_coluna(
            df,
            ['valor']
        )

        if col_valor is None:
            print(
                f"  ⚠ Pulando '{nome_arquivo_fatura}': "
                f"não encontrei coluna de valor."
            )
            continue

        match = re.search(
            r'Fatura(\d{4})-(\d{2})',
            nome_arquivo_fatura
        )

        if not match:
            print(
                f"  ⚠ Não foi possível identificar "
                f"ano/mês de {nome_arquivo_fatura}"
            )
            continue

        ano = int(match.group(1))
        mes = int(match.group(2))

        df['mes_ano'] = f"{mes:02d}/{ano}"
        df['data_fatura'] = pd.Timestamp(
            ano,
            mes,
            1
        )
        df['ano'] = ano
        df['mes'] = mes

        df['valor'] = limpar_valor(
            df[col_valor]
        )

        # Somente valores positivos
        df = df[df['valor'] > 0]

        # Exclusões utilizadas na comparação histórica
        if col_tipo is not None:

            df = df[
                ~df[col_tipo]
                .astype(str)
                .str.upper()
                .str.contains(
                    'PAGAMENTO',
                    na=False
                )
            ]

            df = df[
                ~df[col_tipo]
                .astype(str)
                .str.upper()
                .str.contains(
                    'JORNAL',
                    na=False
                )
            ]

            df = df[
                ~df[col_tipo]
                .astype(str)
                .str.upper()
                .str.contains(
                    'CANCELADO',
                    na=False
                )
            ]

        df = df.dropna(
            subset=['valor']
        )

        df['estabelecimento'] = (
            df[col_estab]
            if col_estab is not None
            else 'NÃO IDENTIFICADO'
        )

        df['tipo'] = (
            df[col_tipo]
            if col_tipo is not None
            else 'OUTROS'
        )

        df['categoria'] = df['tipo']

        lista_faturas_2025.append(df)

        print(
            f"  ✅ {nome_arquivo_fatura}: "
            f"{len(df)} transações — "
            f"{formatar_valor_br(df['valor'].sum())}"
        )

    except Exception as e:

        print(
            f"  ⚠ Não foi possível carregar "
            f"{nome_arquivo_fatura}: "
            f"{str(e)[:100]}"
        )

        continue

df_cartao_2025 = pd.concat(lista_faturas_2025, ignore_index=True) if lista_faturas_2025 else pd.DataFrame()
total_cartao_2025 = df_cartao_2025['valor'].sum() if len(df_cartao_2025) else 0
print(f"\n✅ TOTAL CARTÃO 2025: {len(df_cartao_2025)} transações — {formatar_valor_br(total_cartao_2025)}")

# Despesas da casa 2025
arquivos_casa_2025 = [f for f in arquivos if f.startswith('despesas-casa-') and '2025' in f]
lista_casa_2025 = []
for arq in arquivos_casa_2025:
    try:
        caminho = os.path.join(pasta, arq)
        df = pd.read_csv(caminho, sep=';', encoding='utf-8-sig')
        df.columns = df.columns.str.strip()

        match = re.search(r'(\d{4})-(\d{2})', arq)
        ano, mes = int(match.group(1)), int(match.group(2))
        df['mes_ano'] = f"{mes:02d}/{ano}"
        df['ano'] = ano
        df['mes'] = mes

        df['valor'] = df['Valor'].astype(str).str.replace(',', '.').str.strip()
        df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
        df = df[df['valor'] > 0].dropna(subset=['valor'])

        df['categoria'] = df['Categoria']
        df['estabelecimento'] = df['Descrição']
        df['tipo'] = df['Categoria']

        lista_casa_2025.append(df)
        print(f"  ✅ {arq}: {len(df)} despesas — {formatar_valor_br(df['valor'].sum())}")
    except Exception as e:
        print(f"  ⚠ Não foi possível carregar {arq}: {str(e)[:50]}")
        continue

df_casa_2025 = pd.concat(lista_casa_2025, ignore_index=True) if lista_casa_2025 else pd.DataFrame()
total_casa_2025 = df_casa_2025['valor'].sum() if len(df_casa_2025) else 0
total_geral_2025 = total_cartao_2025 + total_casa_2025
print(f"\n✅ TOTAL CASA 2025: {len(df_casa_2025)} despesas — {formatar_valor_br(total_casa_2025)}")
print(f"💰 TOTAL GERAL 2025: {formatar_valor_br(total_geral_2025)}")

# Gastos mensais 2025
gastos_cartao_mensal_2025 = df_cartao_2025.groupby('mes_ano')['valor'].sum() if len(df_cartao_2025) else pd.Series(dtype=float)
gastos_casa_mensal_2025 = df_casa_2025.groupby('mes_ano')['valor'].sum() if len(df_casa_2025) else pd.Series(dtype=float)

todos_meses_2025 = sorted(set(gastos_cartao_mensal_2025.index) | set(gastos_casa_mensal_2025.index))
gastos_mensais_2025 = pd.DataFrame(index=todos_meses_2025, columns=['cartao', 'casa', 'total'], dtype=float)
for mes in todos_meses_2025:
    cartao = gastos_cartao_mensal_2025.get(mes, 0)
    casa = gastos_casa_mensal_2025.get(mes, 0)
    gastos_mensais_2025.loc[mes] = [cartao, casa, cartao + casa]
if len(gastos_mensais_2025) > 0:
    gastos_mensais_2025['acumulado_total'] = gastos_mensais_2025['total'].cumsum()

print("\n📊 Dados de 2025 carregados com sucesso!")
print(f"   📅 Meses disponíveis: {', '.join(gastos_mensais_2025.index) if len(gastos_mensais_2025) > 0 else 'Nenhum'}")

# ============================================================
# 7. PROJEÇÕES
# ============================================================
print("\n" + "="*60)
print("📈 CALCULANDO PROJEÇÕES PARA 2026...")
print("="*60)

meses_2026 = sorted(set(
    f"{m:02d}/2026" for m in range(1, 13)
    if f"{m:02d}/2026" in gastos_mensais.index
))

projecao_2026 = pd.DataFrame(index=meses_2026, columns=['media_2025', 'projetado_2026', 'variacao_pct'])

if len(gastos_mensais_2025) > 0 and len(gastos_mensais) > 0:
    for mes_2026 in meses_2026:
        mes_num = int(mes_2026.split('/')[0])
        mes_2025 = f"{mes_num:02d}/2025"

        if mes_2025 in gastos_mensais_2025.index:
            media_2025 = gastos_mensais_2025.loc[mes_2025, 'total']
            projecao_2026.loc[mes_2026, 'media_2025'] = media_2025

            if mes_2026 in gastos_mensais.index:
                real_2026 = gastos_mensais.loc[mes_2026, 'total']
                projecao_2026.loc[mes_2026, 'projetado_2026'] = real_2026
                if media_2025 > 0:
                    projecao_2026.loc[mes_2026, 'variacao_pct'] = ((real_2026 - media_2025) / media_2025) * 100
            else:
                projecao_2026.loc[mes_2026, 'projetado_2026'] = media_2025
                projecao_2026.loc[mes_2026, 'variacao_pct'] = 0

meses_futuros_2026 = sorted(set(
    f"{m:02d}/2026" for m in range(1, 13)
    if f"{m:02d}/2026" not in gastos_mensais.index
))

for mes_futuro in meses_futuros_2026:
    mes_num = int(mes_futuro.split('/')[0])
    mes_2025 = f"{mes_num:02d}/2025"

    if mes_2025 in gastos_mensais_2025.index:
        media_2025 = gastos_mensais_2025.loc[mes_2025, 'total']
        projecao_2026.loc[mes_futuro, 'media_2025'] = media_2025
        projecao_2026.loc[mes_futuro, 'projetado_2026'] = media_2025
        projecao_2026.loc[mes_futuro, 'variacao_pct'] = 0

projecao_2026 = projecao_2026.fillna(0)

total_real_2026 = gastos_mensais['total'].sum() if len(gastos_mensais) > 0 else 0
total_projetado_2026 = projecao_2026['projetado_2026'].sum()

print(f"\n📊 Resumo da Projeção:")
print(f"   💰 Total gasto em 2025: {formatar_valor_br(total_geral_2025)}")
print(f"   📈 Projeção anual 2026: {formatar_valor_br(total_projetado_2026)}")
if total_geral_2025 > 0:
    print(f"   📊 Variação percentual: {((total_projetado_2026 - total_geral_2025) / total_geral_2025 * 100):.1f}%")

print("\n📈 Projeção por mês:")
for mes in sorted(projecao_2026.index):
    media = projecao_2026.loc[mes, 'media_2025']
    projetado = projecao_2026.loc[mes, 'projetado_2026']
    variacao = projecao_2026.loc[mes, 'variacao_pct']
    status = "✅ Realizado" if mes in gastos_mensais.index else "🔮 Projetado"
    print(f"   {mes}: {formatar_valor_br(projetado)} (vs média 2025: {formatar_valor_br(media)}) {status} {f'({variacao:+.1f}%)' if variacao != 0 else ''}")

# ============================================================
# 8. GERAR RELATÓRIO COMPARATIVO
# ============================================================
print("\n" + "="*60)
print("📊 GERANDO RELATÓRIO COMPARATIVO...")
print("="*60)

media_mensal_2025 = total_geral_2025 / 12 if total_geral_2025 > 0 else 0
media_mensal_2026 = total_projetado_2026 / 12 if total_projetado_2026 > 0 else 0

meses_comparativos = sorted(set(
    list(gastos_mensais_2025.index if len(gastos_mensais_2025) > 0 else []) +
    list(gastos_mensais.index if len(gastos_mensais) > 0 else []) +
    list(projecao_2026.index)
))

diferencas = []
for mes in sorted(projecao_2026.index):
    if mes in gastos_mensais.index and len(gastos_mensais) > 0:
        real = gastos_mensais.loc[mes, 'total']
        projetado = projecao_2026.loc[mes, 'projetado_2026']
        diferenca = real - projetado
        if projetado > 0:
            pct = (diferenca / projetado) * 100
        else:
            pct = 0
        diferencas.append({
            'mes': mes,
            'real': real,
            'projetado': projetado,
            'diferenca': diferenca,
            'pct': pct
        })

html_comparativo = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Relatório Comparativo 2025 vs 2026</title>
<style>
* {{ margin:0; padding:0; box-sizing:border-box; }}
body {{ font-family:'Segoe UI',Tahoma,Geneva,Verdana,sans-serif; background:#f0f2f5; min-height:100vh; padding:20px; }}
.container {{ max-width:1400px; margin:0 auto; background:white; padding:30px; border-radius:16px; box-shadow:0 10px 40px rgba(0,0,0,0.08); }}
h1 {{ color:#1a1a2e; border-bottom:3px solid #2E86AB; padding-bottom:15px; margin-bottom:30px; font-size:28px; display:flex; align-items:center; gap:10px; }}
h1::before {{ content:"📊"; }}
h2 {{ color:#1a1a2e; margin:30px 0 20px 0; padding:10px 15px; background:#f8f9fa; border-left:4px solid #2E86AB; border-radius:4px; font-size:20px; }}
.header-info {{ background:linear-gradient(135deg,#f5f7fa 0%,#c3cfe2 100%); padding:15px 20px; border-radius:10px; margin-bottom:30px; display:flex; justify-content:space-between; flex-wrap:wrap; gap:15px; }}
.header-info span {{ font-size:14px; color:#555; }}
.header-info strong {{ color:#2E86AB; }}
.metricas {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:20px; margin:25px 0; }}
.metrica {{ padding:20px; border-radius:12px; text-align:center; box-shadow:0 2px 10px rgba(0,0,0,0.06); }}
.metrica .valor {{ font-size:26px; font-weight:bold; margin-top:6px; font-family:'Courier New',monospace; }}
.metrica .label {{ font-size:13px; opacity:0.8; text-transform:uppercase; letter-spacing:0.5px; }}
.metrica-verde {{ background:#e8f5e9; color:#1a5a3a; }}
.metrica-azul {{ background:#e3f2fd; color:#1a3a5a; }}
.metrica-rosa {{ background:#fce4ec; color:#5a1a2a; }}
.metrica-roxo {{ background:#f3e5f5; color:#3a1a4a; }}
.metrica-dourado {{ background:#fff8e1; color:#5a4a1a; }}
.metrica-vermelho {{ background:#ffebee; color:#5a1a1a; }}
.grid-2 {{ display:grid; grid-template-columns:1fr 1fr; gap:25px; margin:25px 0; }}
@media (max-width:768px) {{ .grid-2 {{ grid-template-columns:1fr; }} }}
.info-box {{ background:#f8f9fa; padding:18px 22px; border-radius:10px; margin:15px 0; border-left:4px solid #2E86AB; }}
.info-box strong {{ color:#2E86AB; }}
.tabela-comparativa {{ width:100%; border-collapse:collapse; margin:20px 0; font-size:14px; }}
.tabela-comparativa th {{ background:#2E86AB; color:white; padding:12px 15px; text-align:left; }}
.tabela-comparativa td {{ padding:10px 15px; border-bottom:1px solid #e9ecef; }}
.tabela-comparativa tr:hover {{ background:#f8f9fa; }}
.tabela-comparativa .positivo {{ color:#27ae60; font-weight:bold; }}
.tabela-comparativa .negativo {{ color:#e74c3c; font-weight:bold; }}
.tabela-comparativa .sem-dados {{ color:#95a5a6; }}
.tabela-comparativa .projetado {{ color:#2980b9; font-style:italic; }}
.footer {{ text-align:center; margin-top:40px; padding:20px; background:#f8f9fa; border-radius:10px; color:#7f8c8d; font-size:13px; }}
.footer strong {{ color:#2E86AB; }}
.insights {{ background:#e8f5e9; padding:15px 20px; border-radius:10px; margin:15px 0; border-left:4px solid #27ae60; }}
.insights li {{ margin:6px 0; list-style:none; }}
.insights li::before {{ content:"💡 "; }}
.legenda-tabela {{ margin-top:10px; padding:10px; background:#f8f9fa; border-radius:8px; font-size:12px; color:#7f8c8d; }}
.legenda-tabela span {{ margin-right:15px; }}
</style>
</head>
<body>
<div class="container">
    <h1>📊 RELATÓRIO COMPARATIVO 2025 vs 2026</h1>
    <div class="header-info">
        <span>📅 <strong>Gerado em:</strong> {data_formatada} às {hora_formatada}</span>
        <span>📊 <strong>Período:</strong> 2025 - 2026</span>
        <span>📁 <strong>Análise:</strong> Comparativo e Projeções</span>
    </div>

    <div class="metricas">
        <div class="metrica metrica-verde"><div class="label">💰 Total 2025</div><div class="valor">{formatar_valor_br(total_geral_2025)}</div></div>
        <div class="metrica metrica-azul"><div class="label">💰 Total 2026</div><div class="valor">{formatar_valor_br(total_projetado_2026)}</div></div>
        <div class="metrica metrica-roxo"><div class="label">📊 Média Mensal 2025</div><div class="valor">{formatar_valor_br(media_mensal_2025)}</div></div>
        <div class="metrica metrica-dourado"><div class="label">📊 Média Mensal 2026</div><div class="valor">{formatar_valor_br(media_mensal_2026)}</div></div>
    </div>

    <h2>📊 COMPARATIVO MENSAL - 2025 vs 2026</h2>
    <div style="overflow-x:auto;">
    <table class="tabela-comparativa">
        <thead>
            <tr>
                <th>Mês</th>
                <th>Gasto 2025</th>
                <th>Gasto 2026</th>
                <th>Diferença (R$)</th>
                <th>Variação (%)</th>
                <th>Status</th>
            </tr>
        </thead>
        <tbody>
"""

for mes in sorted(projecao_2026.index):
    gasto_2025 = projecao_2026.loc[mes, 'media_2025']
    gasto_2026_real = projecao_2026.loc[mes, 'projetado_2026']
    variacao = projecao_2026.loc[mes, 'variacao_pct']

    mes_realizado = mes in gastos_mensais.index

    if mes_realizado:
        status = "✅ Realizado"
        cor = "positivo" if variacao > 0 else "negativo" if variacao < 0 else ""
        cel_gasto_2026 = formatar_valor_br(gasto_2026_real)
        cel_diferenca = f'<span class="{cor}">{formatar_valor_br(gasto_2026_real - gasto_2025)}</span>'
        cel_variacao = f'<span class="{cor}">{variacao:+.1f}%</span>'
    else:
        status = f'🔮 Projeção: {formatar_valor_br(gasto_2026_real)}'
        cel_gasto_2026 = '<span style="color:#aaa;">—</span>'
        cel_diferenca = '<span style="color:#aaa;">—</span>'
        cel_variacao = '<span style="color:#aaa;">—</span>'

    html_comparativo += f"""
            <tr>
                <td><strong>{mes}</strong></td>
                <td>{formatar_valor_br(gasto_2025)}</td>
                <td>{cel_gasto_2026}</td>
                <td>{cel_diferenca}</td>
                <td>{cel_variacao}</td>
                <td>{status}</td>
            </tr>
    """

html_comparativo += f"""
        </tbody>
    </table>
    </div>

    <div class="legenda-tabela">
        📌 <strong>Legenda:</strong>
        <span style="color:#27ae60;">✅ Realizado</span> — mês com dados processados da fatura
        <span style="color:#2980b9;">🔮 Projetado</span> — mês sem dados, baseado na média de 2025
        <span style="color:#27ae60;">▲</span> Aumento em relação a 2025
        <span style="color:#e74c3c;">▼</span> Redução em relação a 2025
    </div>

    <h2>📈 ANÁLISE DE PROJEÇÃO</h2>
    <div class="grid-2">
        <div class="info-box" style="border-left-color:#27ae60;background:#e8f5e9;">
            <h3 style="color:#27ae60;">✅ Valores Realizados 2026</h3>
            <p><strong>Total realizado 2026:</strong> {formatar_valor_br(total_real_2026)}</p>
            <p><strong>Meses com dados:</strong> {len(gastos_mensais)}</p>
        </div>
        <div class="info-box" style="border-left-color:#e74c3c;background:#fde8e8;">
            <h3 style="color:#e74c3c;">🔮 Projeção para 2026</h3>
            <p><strong>Projeção total 2026:</strong> {formatar_valor_br(total_projetado_2026)}</p>
            <p><strong>Variação anual:</strong> {formatar_valor_br(total_projetado_2026 - total_geral_2025)}</p>
            <p><strong>Variação percentual:</strong> {((total_projetado_2026 - total_geral_2025) / total_geral_2025 * 100 if total_geral_2025 > 0 else 0):.1f}%</p>
        </div>
    </div>

    <h2>💡 INSIGHTS COMPARATIVOS</h2>
    <div class="insights">
        <ul>
"""

if total_geral_2025 > 0 and total_projetado_2026 > 0:
    variacao_total = ((total_projetado_2026 - total_geral_2025) / total_geral_2025) * 100
    if variacao_total > 5:
        html_comparativo += f"<li>📈 Projeção de aumento de {variacao_total:.1f}% em relação a 2025</li>"
    elif variacao_total < -5:
        html_comparativo += f"<li>📉 Projeção de redução de {abs(variacao_total):.1f}% em relação a 2025</li>"
    else:
        html_comparativo += f"<li>📊 Projeção estável com variação de {variacao_total:.1f}% em relação a 2025</li>"

if diferencas:
    maiores_aumentos = sorted(diferencas, key=lambda x: x['pct'], reverse=True)[:2]
    maiores_reducoes = sorted(diferencas, key=lambda x: x['pct'])[:2]

    for item in maiores_aumentos:
        if item['pct'] > 0:
            html_comparativo += f"<li>📈 {item['mes']}: aumento de {item['pct']:.1f}% (R$ {formatar_valor_br(item['diferenca'])})</li>"

    for item in maiores_reducoes:
        if item['pct'] < 0:
            html_comparativo += f"<li>📉 {item['mes']}: redução de {abs(item['pct']):.1f}% (R$ {formatar_valor_br(item['diferenca'])})</li>"

# ============================================================
# ANÁLISE DE DESPESAS RELEVANTES - RELATÓRIO COMPARATIVO
# ============================================================
from html import escape as escapar_html
import unicodedata

def normalizar_categoria(texto):
    texto = unicodedata.normalize('NFKD', str(texto or '').strip().upper())
    return ''.join(c for c in texto if not unicodedata.combining(c))

# Os nomes abaixo correspondem às categorias efetivas da planilha.
# CASA é diferente de CONTAS DA CASA (lançamentos do arquivo de despesas domésticas).
grupos_analise = [
    'MERCADO', 'FARMÁCIA', 'COMBUSTÍVEL', 'ALMOÇO SENAC',
    'CASA', 'MONK / NINA', 'ESTACIONAMENTO', 'PERFUMARIA',
    'SEGURO DO CARRO', 'VESTUÁRIO', 'BILHETE ÚNICO',
    'CONTAS DA CASA', 'MECÂNICO / CARRO', 'TÁXI',
    'ALIMENTAÇÃO', 'SAÚDE / DENTISTA', 'OUTROS'
]
mapa_grupos = {
    'MERCADO':'MERCADO', 'FARMACIA':'FARMÁCIA',
    'COMBUSTIVEL':'COMBUSTÍVEL', 'ALMOCO SENAC':'ALMOÇO SENAC',
    'CASA':'CASA', 'MONK':'MONK / NINA', 'NINA':'MONK / NINA',
    'ESTACIONAMENTO':'ESTACIONAMENTO', 'PERFUMARIA':'PERFUMARIA',
    'PERFURMARIA':'PERFUMARIA', 'SEGURO DO CARRO':'SEGURO DO CARRO',
    'VESTUARIO':'VESTUÁRIO', 'BILHETE UNICO':'BILHETE ÚNICO',
    'MECANICO':'MECÂNICO / CARRO', 'CARRO':'MECÂNICO / CARRO',
    'TAXI':'TÁXI', 'ALIMENTACAO':'ALIMENTAÇÃO',
    'SAUDE':'SAÚDE / DENTISTA', 'DENTISTA':'SAÚDE / DENTISTA'
}

# Usa apenas lançamentos positivos; reembolsos de trabalho são informativos,
# mas não compõem o orçamento pessoal por categoria.
base_analise = df_total_mensal.copy()
base_analise['valor'] = pd.to_numeric(base_analise['valor'], errors='coerce').fillna(0)
base_analise = base_analise[base_analise['valor'] > 0].copy()
base_analise['mes_ano'] = base_analise['mes_ano'].astype(str)
base_analise = base_analise[base_analise['mes_ano'].str.match(r'^\d{1,2}/2026$')].copy()
base_analise['cat_normalizada'] = base_analise['categoria'].map(normalizar_categoria)
excluir_analise = {'PAGAMENTO','CANCELADO','JORNAL','VIAGEM TRABALHO REEMBOLSO','TRABALHO REEMBOLSO'}
base_analise = base_analise[~base_analise['cat_normalizada'].isin(excluir_analise)].copy()
base_analise['grupo'] = base_analise['cat_normalizada'].map(mapa_grupos).fillna('OUTROS')

# Contas da casa: categorias oriundas de arquivos de despesas domésticas,
# identificadas separadamente do cartão para não somar a mesma linha duas vezes.
if len(df_casa):
    base_casa = df_casa[['mes_ano','categoria','valor']].copy()
    base_casa['valor'] = pd.to_numeric(base_casa['valor'],errors='coerce').fillna(0)
    base_casa['mes_ano'] = base_casa['mes_ano'].astype(str)
    base_casa = base_casa[(base_casa['valor'] > 0) & base_casa['mes_ano'].str.match(r'^\d{1,2}/2026$')]
    # Remover do consolidado a contribuição da casa por mês/categoria/valor
    # não é seguro se houver valores idênticos; em vez disso reconstruímos a base.
    base_cartao = df_cartao_2026[['mes_ano','categoria','valor']].copy() if len(df_cartao_2026) else pd.DataFrame(columns=['mes_ano','categoria','valor'])
    base_cartao['valor'] = pd.to_numeric(base_cartao['valor'],errors='coerce').fillna(0)
    base_cartao['mes_ano'] = base_cartao['mes_ano'].astype(str)
    base_cartao = base_cartao[(base_cartao['valor'] > 0) & base_cartao['mes_ano'].str.match(r'^\d{1,2}/2026$')].copy()
    base_cartao['cat_normalizada'] = base_cartao['categoria'].map(normalizar_categoria)
    base_cartao = base_cartao[~base_cartao['cat_normalizada'].isin(excluir_analise)]
    base_cartao['grupo'] = base_cartao['cat_normalizada'].map(mapa_grupos).fillna('OUTROS')
    base_casa['grupo'] = 'CONTAS DA CASA'
    base_analise = pd.concat([base_cartao[['mes_ano','valor','grupo']],base_casa[['mes_ano','valor','grupo']]],ignore_index=True)

analise_categorias_html = '<h2>📊 ANÁLISE DAS DESPESAS ESSENCIAIS — 2026</h2>'
analise_categorias_html += ('<p style="font-size:13px;color:#555;margin-bottom:15px">'
    'Categorias relevantes para o orçamento. Outras despesas ficam agrupadas em OUTROS. '
    'Contas da casa são exibidas em um bloco próprio; reembolsos de trabalho não entram neste ranking. '
    'Os meses exibidos correspondem às faturas disponíveis, não a projeções.</p>')

if len(base_analise):
    meses_analise = sorted(base_analise['mes_ano'].unique(),key=lambda x:int(x.split('/')[0]))
    resumo_grupos = base_analise.groupby('grupo')['valor'].sum().sort_values(ascending=False)
    total_analise = float(resumo_grupos.sum())
    maior_total = float(resumo_grupos.max()) if len(resumo_grupos) else 1
    analise_categorias_html += '<h3>Ranking de gastos e média mensal</h3>'
    for grupo,total in resumo_grupos.items():
        largura = round(float(total)/maior_total*100,1) if maior_total else 0
        media = float(total)/len(meses_analise)
        percentual = 100*float(total)/total_analise if total_analise else 0
        analise_categorias_html += f"""
        <div style="margin:9px 0;padding:11px 13px;background:#f8f9fa;border-radius:8px">
          <div style="display:flex;justify-content:space-between;gap:8px;flex-wrap:wrap;font-size:13px">
            <strong>{escapar_html(grupo)}</strong>
            <span>Total {formatar_valor_br(total)} · Média {formatar_valor_br(media)}/mês · {percentual:.1f}%</span>
          </div>
          <div style="height:11px;background:#e1e7ec;border-radius:8px;margin-top:8px;overflow:hidden">
            <div style="height:100%;width:{largura}%;background:#2E86AB"></div>
          </div>
        </div>"""
    analise_categorias_html += '<h3 style="margin-top:26px">Em que mês cada despesa foi maior?</h3>'
    analise_categorias_html += '<p style="font-size:12px;color:#666">Barras verdes indicam o maior valor mensal de cada grupo. Meses sem gasto aparecem zerados.</p>'
    por_mes_grupo = base_analise.groupby(['grupo','mes_ano'])['valor'].sum()
    for grupo in grupos_analise:
        if grupo not in resumo_grupos.index: continue
        valores = {mes:float(por_mes_grupo.get((grupo,mes),0)) for mes in meses_analise}
        pico = max(valores.values())
        mes_pico = next(mes for mes,valor in valores.items() if valor == pico)
        analise_categorias_html += f"""
        <div style="margin:15px 0;padding:14px;border:1px solid #e2e8ed;border-radius:10px">
          <div style="display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap;margin-bottom:10px">
            <strong>{escapar_html(grupo)}</strong>
            <span style="font-size:12px;color:#24734b">Pico: {nome_mes_pt(int(mes_pico.split('/')[0]))} — {formatar_valor_br(pico)}</span>
          </div>
          <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(92px,1fr));gap:7px">"""
        for mes,valor in valores.items():
            destaque = valor == pico and pico > 0
            altura = max(2,round(valor/pico*58)) if pico else 2
            cor = '#27ae60' if destaque else '#2E86AB'
            fundo = '#e8f5e9' if destaque else '#f8f9fa'
            analise_categorias_html += f"""
            <div style="background:{fundo};border-radius:7px;padding:8px 4px;text-align:center">
              <div style="height:62px;display:flex;align-items:flex-end;justify-content:center">
                <div style="width:28px;height:{altura}px;background:{cor};border-radius:4px 4px 0 0"></div>
              </div>
              <div style="font-size:11px;margin-top:5px">{nome_mes_pt(int(mes.split('/')[0]))[:3]}</div>
              <div style="font-size:11px;font-weight:600">{formatar_valor_br(valor)}</div>
            </div>"""
        analise_categorias_html += '</div></div>'
else:
    analise_categorias_html += '<p>Sem dados disponíveis para esta análise.</p>'

html_comparativo += f"""
        </ul>
    </div>

    {analise_categorias_html}

    <div class="footer">
        <p>📅 Relatório comparativo gerado em <strong>{data_formatada} às {hora_formatada}</strong></p>
        <p>📊 Dados baseados em faturas de cartão de crédito e despesas da casa</p>
        <p>🔮 Projeções baseadas na média do mesmo mês do ano anterior (2025)</p>
        <p style="color:#95a5a6;font-size:11px;margin-top:5px;">✅ Comparativo automático entre anos consecutivos</p>
    </div>
</div>
</body>
</html>
"""

nome_arquivo_comparativo = f"relatorio_comparativo_2025_2026_{data_hora_atual.strftime('%Y%m%d_%H%M')}.html"
caminho_comparativo = os.path.join(pasta_relatorios, nome_arquivo_comparativo)
with open(caminho_comparativo, 'w', encoding='utf-8') as f:
    f.write(html_comparativo)

print(f"\n✅ Relatório comparativo salvo em: {caminho_comparativo}")

files.download(caminho_comparativo)
print("⬇️  Download do relatório comparativo iniciado")

print("\n" + "="*60)
print("✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Encontrados 23 arquivos na pasta de Faturas
📁 Encontrados 18 arquivos na pasta de Casa (/content/drive/MyDrive/Contas/Casa)
   📄 despesas-casa-2025-01.xlsx
   📄 despesas-casa-2025-02.xlsx
   📄 despesas-casa-2025-03.xlsx
   📄 despesas-casa-2025-04.xlsx
   📄 despesas-casa-2025-05.xlsx
   📄 despesas-casa-2025-06.xlsx
   📄 despesas-casa-2025-07.xlsx
   📄 despesas-casa-2025-08.xlsx
   📄 despesas-casa-2025-09.xlsx
   📄 despesas-casa-2025-10.xlsx
   📄 despesas-casa-2026-01.xlsx
   📄 despesas-casa-2026-03.xlsx
   📄 despesas-casa-2026-05.xlsx
   📄 despesas-casa-2026-06.xlsx
   📄 despesas-casa-2026-07.xlsx
   📄 despesas-casa-2026-08.xlsx
   📄 despesas-casa-2026-09.xlsx
   📄 despesas-casa-2026-10.xlsx

📋 ARQUIVO DE FATURAS
📄 Arquivo: Faturas.xlsx
📊 Registros encontrados: 1804

📁 Faturas 2025 encontradas: 12
   📄 Fatura2025-01-05.csv
   📄 Fatura2025-02-05.csv
   📄 Fatu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download do relatório 2026 iniciado

📊 CARREGANDO DADOS DE 2025 PARA COMPARAÇÃO...

📄 Processando: Fatura2025-01-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-01-05.csv: 68 transações — R$ 4.896,66

📄 Processando: Fatura2025-02-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-02-05.csv: 65 transações — R$ 4.534,28

📄 Processando: Fatura2025-03-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-03-05.csv: 70 transações — R$ 4.974,58

📄 Processando: Fatura2025-04-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-04-05.csv: 69 transações — R$ 4.899,66

📄 Processando: Fatura2025-05-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-05-05.csv: 61 transações — R$ 5.448,15

📄 Processando: Fatura2025-06-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-06-05.csv: 76 transações — R$ 5.673,82

📄 Processando: Fatura2025-07-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-07-05.csv: 87 transações — R$ 6.921,61

📄 Processando: Fatura2025-08-05.csv  [Faturas.xlsx]
  ✅ Fatura2025-08-05.csv: 82 transações — R$ 5.667,97

📄 Processando: Fatura2025-09-05.csv  [Faturas.xlsx]
  ✅ 

/tmp/ipykernel_1467/837867199.py:2172: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  projecao_2026 = projecao_2026.fillna(0)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download do relatório comparativo iniciado

✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!
